In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Environment Ready")
occupancy = pd.read_csv("../data/urbanev/occupancy.csv")
volume = pd.read_csv("../data/urbanev/volume.csv")
price = pd.read_csv("../data/urbanev/price.csv")
duration = pd.read_csv("../data/urbanev/duration.csv")

stations = pd.read_csv("../data/urbanev/stations.csv")
time = pd.read_csv("../data/urbanev/time.csv")
information = pd.read_csv("../data/urbanev/information.csv")
distance = pd.read_csv("../data/urbanev/distance.csv")
adj = pd.read_csv("../data/urbanev/adj.csv")

acn = pd.read_excel("../data/acn/acndata_sessions.json.xlsx")

In [ ]:
datasets = {
    "occupancy": occupancy,
    "volume": volume,
    "price": price,
    "duration": duration,
    "stations": stations,
    "time": time,
    "information": information,
    "distance": distance,
    "adj": adj,
    "acn": acn
}

for name, df in datasets.items():
    print("\n" + "="*60)
    print(name.upper())
    print("="*60)
    print("Shape:", df.shape)

In [ ]:
occupancy.head()

In [ ]:
volume.head()

In [ ]:
price.head()

In [ ]:
duration.head()

In [ ]:
stations.head()

In [ ]:
information.head()

In [ ]:
time.head()

In [ ]:
for name, df in datasets.items():
    print("\n")
    print(name)
    print(df.columns.tolist())

In [ ]:
missing = []

for name, df in datasets.items():
    missing.append([
        name,
        df.isnull().sum().sum()
    ])

missing_df = pd.DataFrame(
    missing,
    columns=["dataset","missing_values"]
)

missing_df

In [ ]:
missing_acn = (
    acn.isnull()
       .sum()
       .sort_values(ascending=False)
)

missing_acn[missing_acn > 0]

In [ ]:
full_missing = acn.columns[acn.isnull().all()]

print(full_missing.tolist())

In [ ]:
acn.drop(columns=full_missing, inplace=True)

In [ ]:
missing_pct = (
    acn.isnull()
       .mean()
       .sort_values(ascending=False)
       * 100
)

missing_pct = missing_pct[missing_pct > 0]

missing_pct

In [ ]:
acn["userID"].nunique()

In [ ]:
acn.columns.tolist()

In [ ]:
acn.info()

In [ ]:
drop_cols = [
    "site",                 # 99.99% missing
    "userID",               # 86% missing
    "WhPerMile",            # 78% missing
    "kWhRequested",         # 78% missing
    "milesRequested",       # 78% missing
    "minutesAvailable",     # 78% missing
    "modifiedAt",           # 78% missing
    "paymentRequired",      # 78% missing
    "requestedDeparture",   # 78% missing
    "userID.1"              # duplicate user field
]

acn = acn.drop(columns=drop_cols)

In [ ]:
date_cols = [
    "connectionTime",
    "disconnectTime",
    "doneChargingTime"
]

for col in date_cols:
    acn[col] = pd.to_datetime(acn[col], errors="coerce")

acn.info()

In [ ]:
acn["session_duration_hours"] = (
    acn["disconnectTime"] -
    acn["connectionTime"]
).dt.total_seconds() / 3600

In [ ]:
acn["charging_duration_hours"] = (
    acn["doneChargingTime"] -
    acn["connectionTime"]
).dt.total_seconds() / 3600

In [ ]:
acn[
    [
        "session_duration_hours",
        "charging_duration_hours",
        "kWhDelivered"
    ]
].describe()

In [ ]:
acn["hour"] = acn["connectionTime"].dt.hour
acn["dayofweek"] = acn["connectionTime"].dt.dayofweek
acn["month"] = acn["connectionTime"].dt.month

In [ ]:
acn[
    [
        "hour",
        "dayofweek",
        "month"
    ]
].head()

In [ ]:
acn.to_csv(
    "../data/acn/acn_cleaned.csv",
    index=False
)

print("ACN cleaned dataset saved.")

# Key Findings

## UrbanEV Dataset

- Contains 247 charging grids observed over 8640 time intervals.
- No missing values were found.
- Occupancy, volume, duration, and price share identical structures.
- Data is recorded at 5-minute intervals.

## ACN Dataset

- Contains 16,304 charging sessions.
- Missing values are concentrated in  user-preference fields.
- Core operational charging variables remain largely complete.
- Datetime variables were converted and session-level features were created.

## Preprocessing Decisions

- Removed columns with excessive missingness (>75%).
- Retained operational charging session fields.
- Generated charging duration and session duration features.
- Saved cleaned dataset for downstream modeling.

## Feature Retention Rationale (ACN)

- `connectionTime`, `disconnectTime`, `doneChargingTime`: used to derive session duration, 
  charging duration, and idle time — core to pricing and utilization analysis.
- `kWhDelivered`: the revenue variable. Essential for all tariff revenue calculations.
- `stationID`, `clusterID`: station-level grouping for utilization analysis.
- `hour`, `dayofweek`, `month`: temporal features for demand prediction.
- Dropped columns (>75% missing) are excluded because imputation would introduce 
  systematic bias into revenue and behavioral estimates.